# Monitoramento multimodal — Tech Challenge Fase 4

Este é o ponto de entrada da demonstração. Áudio, vídeo, texto e séries estruturadas convergem para uma caixa de revisão. As modalidades são associadas para o experimento; não representam um paciente pareado.

A execução padrão utiliza resultados reais já registrados dos modelos e da nuvem. A fusão, os detectores estruturados, a persistência e o painel são executados novamente. Não há treinamento ou chamadas pagas.


In [1]:
from pathlib import Path
import json, sys, subprocess

# A mesma entrada funciona na raiz, em subpastas locais e no Google Colab.
candidatos = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "TechChallengeFase4"]
REPO = next((p for p in candidatos if (p / "src/techchallenge4").is_dir()), None)
if REPO is None:
    if not Path("/content").is_dir():
        raise FileNotFoundError("Abra o notebook dentro do repositório TechChallengeFase4.")
    REPO = Path("/content/TechChallengeFase4")
    if not REPO.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--branch",
                "Nirton",
                "--single-branch",
                "https://github.com/Val-Faria/TechChallengeFase4.git",
                str(REPO),
            ],
            check=True,
        )
sys.path.insert(0, str(REPO / "src"))
from techchallenge4.notebook_display import display, HTML

print("Projeto: TechChallengeFase4 | execução local ou Google Colab")

Projeto: TechChallengeFase4 | execução local ou Google Colab


In [2]:
import pandas as pd
from techchallenge4.demo import run_demo

SAIDA = REPO / "integracao/saidas/apresentacao_final"
resumo = run_demo(
    SAIDA,
    audio_csv=REPO / "audio/evidencias/resultado_audio_externo.csv",
    audio_filename="novo-03.wav",
    video_json=REPO / "evidence/resultado_video_teste.json",
    text_json=REPO / "evidence/texto_openai_integracao.json",
)
print(
    "Casos:",
    resumo["resultados"],
    "| Na revisão:",
    resumo["caixa_revisao"],
    "| Entrega idempotente:",
    resumo["idempotencia"],
)

Casos: 5 | Na revisão: 4 | Entrega idempotente: True


## 1. Fontes e decisão do caso

A confiança acústica abaixo de 0,70 permanece inconclusiva. A fusão conserva a maior prioridade dos sinais disponíveis; uma modalidade neutra não cancela um alerta alto. Cobertura parcial também exige revisão, mesmo sem alerta positivo.


In [3]:
caso = json.loads((SAIDA / "demonstracao-integrada.json").read_text(encoding="utf-8"))
display(
    pd.DataFrame(
        [
            {
                "fonte": f["modalidade"],
                "estado": f["status"],
                "alerta": f["alerta"],
                "nível": f["nivel"],
                "cobertura_parcial": f.get("cobertura_parcial", False),
            }
            for f in caso["fontes"]
        ]
    )
)
print(
    "Prioridade final:",
    caso["nivel"],
    "| Cobertura:",
    caso["status"],
    "| Entrega:",
    caso["entrega"]["status"],
)
assert caso["nivel"] == "alto" and caso["requer_revisao"]

Prioridade final: alto | Cobertura: incompleto | Entrega: persistido


fonte,estado,alerta,nível,cobertura_parcial
audio,baixa_confianca,False,nenhum,False
video,ok,True,alto,True
sinais_vitais,ok,True,medio,False
movimentacao,ok,True,medio,False
prescricoes,ok,True,medio,False
texto_openai,ok,True,medio,False


## 2. Painel da caixa de revisão

In [4]:
display(HTML((SAIDA / "painel.html").read_text(encoding="utf-8")))
print("Para confirmar uma revisão, iniciar o painel local sobre o banco desta saída.")

Para confirmar uma revisão, iniciar o painel local sobre o banco desta saída.


Painel de revisão multimodal

## 3. Monitoramento incremental

Trinta amostras sintéticas formam o histórico anterior. Os próximos pacotes verificam uma observação de referência e alterações em sinais vitais, prescrições e movimento. A latência mede processamento e entrega locais, sem nuvem, captura de sensores ou inferência dos modelos.


In [5]:
from techchallenge4.monitor import example_packets, run_stream
from techchallenge4.reports import atomic_write

FLUXO = SAIDA / "monitoramento"
entrada = FLUXO / "entrada_sintetica.jsonl"
atomic_write(entrada, "".join(json.dumps(p, ensure_ascii=False) + "\n" for p in example_packets()))
monitoramento = run_stream(entrada, FLUXO, max_events=32)
print(json.dumps(monitoramento, ensure_ascii=False, indent=2))
display(HTML((FLUXO / "painel.html").read_text(encoding="utf-8")))

{
  "pacotes_processados": 32,
  "casos": 1,
  "alertas_caixa": 1,
  "modo": "arquivo_jsonl_incremental",
  "limite_eventos": 32,
  "timeout_ociosidade_s": 5.0,
  "escopo": "processamento_incremental_local_sem_sensores_nem_inferencia_modelos",
  "latencias_ms": [
    0.2795000000332948,
    0.25829999867710285,
    0.26389999766252004,
    0.27550000231713057,
    0.29519999952754006,
    0.2944999978353735,
    0.34159999995608814,
    0.6171999993966892,
    0.33540000003995374,
    0.34919999961857684,
    0.35370000114198774,
    0.3724999987753108,
    0.37800000063725747,
    0.38560000029974617,
    0.39939999987836927,
    0.5831000016769394,
    0.42320000284234993,
    0.43419999929028563,
    0.9068999970622826,
    0.47839999751886353,
    0.49529999887454323,
    0.49339999895892106,
    0.5070999977760948,
    0.5312000030244235,
    0.5290000008244533,
    0.538200001756195,
    0.5613000030280091,
    0.5662000003212597,
    0.6008999989717267,
    0.5983999981253874,
 

Painel de revisão multimodal

## 4. Transcrição e análise textual

A fonte seguinte é a resposta real da API obtida na avaliação anterior. Sua leitura não realiza outra solicitação. Novas chamadas são opcionais, usam `OPENAI_API_KEY` nos Secrets do Colab e respeitam o controle de orçamento descrito no repositório.


In [6]:
texto = json.loads((REPO / "evidence/texto_openai_integracao.json").read_text(encoding="utf-8"))
print(json.dumps(texto, ensure_ascii=False, indent=2))
print("Chamadas pagas nesta execução: 0")

{
  "modalidade": "texto_openai",
  "status": "ok",
  "alerta": true,
  "nivel": "medio",
  "motivo": "2 menção(ões) afirmada(s) atuais; 0 incerta(s).",
  "origem": "openai_responses",
  "simulado": false,
  "termos_criticos": [
    "falta de ar",
    "tontura"
  ],
  "sentimento": "negativo",
  "mencoes": [
    {
      "termo": "falta de ar",
      "estado": "afirmado",
      "contexto": "paciente_atual",
      "evidencia": "Desde o começo da tarde sinto falta de ar e uma tontura que não passa."
    },
    {
      "termo": "tontura",
      "estado": "afirmado",
      "contexto": "paciente_atual",
      "evidencia": "Desde o começo da tarde sinto falta de ar e uma tontura que não passa."
    }
  ],
  "limitacoes": [
    "Incerteza sobre a gravidade dos sintomas."
  ],
  "ressalva": "Extração textual para revisão; sentimento não é diagnóstico ou gravidade.",
  "modelo": "gpt-4o-mini",
  "id_resposta": "resp_0e98257b9bcd8c1c016af25a69a6bc87d2949c306be77b9b0f",
  "sha256_transcricao": "de

## 5. Salvar os resultados

In [7]:
import zipfile

pacote = SAIDA.parent / "resultados_apresentacao.zip"
with zipfile.ZipFile(pacote, "w", zipfile.ZIP_DEFLATED) as arquivo:
    for p in sorted(SAIDA.rglob("*")):
        if p.is_file() and p.suffix in {".json", ".jsonl", ".csv", ".md", ".html"}:
            arquivo.write(p, p.relative_to(SAIDA))
print("Resultados salvos: integracao/saidas/resultados_apresentacao.zip")
print("Relatório final: output/pdf/relatorio_tecnico_fase4.pdf")

Resultados salvos: integracao/saidas/resultados_apresentacao.zip
Relatório final: output/pdf/relatorio_tecnico_fase4.pdf


O painel é local e a confirmação registra revisão de demonstração. Os resultados não validam decisões clínicas. Consulte o relatório para as partições, as divergências do vídeo e as limitações dos áudios sintéticos. Ao usar GPU opcional no Colab, salve os artefatos e desconecte/exclua o ambiente ao terminar.
